# 04 — Churn Prediction
XGBoost classifier with SHAP explainability.

In [ ]:
import sys; sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost as xgb
from sqlalchemy import create_engine
from src.churn import build_features, FEATURES, FEATURE_LABELS
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, roc_curve
import warnings; warnings.filterwarnings('ignore')
shap.initjs()

engine = create_engine('sqlite:///../data/ecommerce.db')
sns.set_theme(style='whitegrid')
print("Ready")

## 1. Feature Engineering

In [ ]:
df = build_features(engine)
print(f"Customers: {len(df):,}")
print(f"Churn rate: {df['churned'].mean()*100:.1f}%")
df[FEATURES + ['churned']].describe().round(2)

## 2. Feature Correlations with Churn

In [ ]:
corr = df[FEATURES + ['churned']].corr()['churned'].drop('churned').sort_values()
fig, ax = plt.subplots(figsize=(8,8))
colors = ['#e74c3c' if v > 0 else '#2980b9' for v in corr.values]
ax.barh([FEATURE_LABELS.get(f,f) for f in corr.index], corr.values, color=colors)
ax.axvline(0, color='black', lw=1)
ax.set_title('Feature Correlation with Churn Label')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout(); plt.show()

## 3. Train XGBoost

In [ ]:
X = df[FEATURES]; y = df['churned']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
neg,pos = (y_train==0).sum(),(y_train==1).sum()

model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    scale_pos_weight=neg/pos, eval_metric='logloss',
    early_stopping_rounds=20, random_state=42, verbosity=0
)
model.fit(X_train,y_train,eval_set=[(X_test,y_test)],verbose=False)

y_proba = model.predict_proba(X_test)[:,1]
y_pred  = model.predict(X_test)
auc = roc_auc_score(y_test, y_proba)
print(f"AUC-ROC: {auc:.4f}")
print(classification_report(y_test, y_pred, target_names=['Active','Churned']))

## 4. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
fig, ax = plt.subplots(figsize=(7,6))
ax.plot(fpr, tpr, color='#2980b9', lw=2, label=f'XGBoost (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'--',color='gray'); ax.fill_between(fpr,tpr,alpha=0.1,color='#2980b9')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Churn Classifier'); ax.legend()
plt.tight_layout(); plt.show()

## 5. SHAP Explainability

In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10,8))
shap.summary_plot(shap_values, X_test,
                  feature_names=[FEATURE_LABELS.get(f,f) for f in FEATURES],
                  show=False)
plt.title('SHAP Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Score All Customers & Export

In [ ]:
df['churn_probability'] = model.predict_proba(df[FEATURES])[:,1]
df['risk_tier'] = pd.cut(df['churn_probability'],
                          bins=[0,.33,.66,1.], labels=['Low Risk','Medium Risk','High Risk'])

out = df[['customer_id','churned','churn_probability','risk_tier']+FEATURES]
out.to_csv('../outputs/churn_predictions.csv', index=False)

tier_counts = df['risk_tier'].value_counts()
print("Risk tier breakdown:")
print(tier_counts.to_string())
print(f"\nExported {len(out):,} rows → outputs/churn_predictions.csv")